In [ ]:
import cdflib
import numpy as np
import scipy.constants as sc 
import matplotlib.pyplot as plt 
import scipy.interpolate as sci
import pyspedas
from pyspedas.projects.mms import mms_load_feeps
import numpy as np
import datetime as dt
import pandas as pd
from scipy.io import loadmat
cdf_path = "c:/Users/antho/OneDrive/Desktop/Bsc project/AMDA/2017-10-13_08-59-57.cdf" 

read_file = cdflib.CDF(cdf_path) #read cdf
print(read_file.cdf_info()) #print out information about cdf
variable_names = read_file.cdf_info().zVariables #Names of variables
print(variable_names)
time = cdflib.cdfepoch.to_datetime(read_file.varget('AMDA_TIME'))

def dates(times):
    date = str(times.astype("datetime64[D]"))
    time = str(times.astype("datetime64[s]").astype(object).time())
    return date + "/" + time
start = dates(time[0])
end = dates(time[-1])
print(start,end)
feeps_vars = mms_load_feeps(trange=[start, end], datatype='ion',data_rate="brst") #download MMS burst data

In [ ]:
b_gse = read_file.varget('mms1_brst_bgse')
b_tot = read_file.varget('mms1_fgm_brst_b_tot_16288359309142394004')
ion_density = read_file.varget('mms1_fpi_brst_46_dism_mms1_dis_numberdensity_brst_16288359309142394004')
ion_velocity = read_file.varget('mms1_fpi_brst_46_dism_mms1_dis_bulkv_gse_brst_16288359309142394004')
ion_flux = read_file.varget('mms1_fpi_brst_46_dism_mms1_dis_energyspectr_omni_brst_16288359309142394004')
ion_densityxvelocity = ion_velocity * ion_density[np.newaxis,:,np.newaxis] #embeds the data array as a single element of an array
ion_densityxvelocity = ion_densityxvelocity[0] #pulls the data array out of the array

dataset = pyspedas.get_data("mms1_epd_feeps_brst_l2_ion_intensity_omni") #extract usable data from MMS file
FEEPs_time = dataset.times #Epoch time
FEEPs_energies = dataset.v #keV
FEEPs_intensities = dataset.y
FEEPs_time_corr = FEEPs_time.astype("datetime64[s]") #convcert epoch time to numpy.datetime64
def slicing_data(data,start,end):
    """Disect data file from a start and end of time"""
    start_index = np.where(time.astype("datetime64[s]") == np.datetime64(start))[0][0] #array index of start time
    end_index = np.where(time.astype("datetime64[s]") == np.datetime64(end))[0][0] #array index of end time
    return data[start_index:end_index+1] #slice data into range of desired start and end indexes
def average_value(data):
    """Find the mean value/array of the data
    """
    return np.mean(data,axis = 0) 
pyspedas.tplot('mms1_epd_feeps_brst_l2_ion_intensity_omni')

In [ ]:
#upstream and downstream time intervals (user input)
start_u = "2017-10-13T08:59:04"
end_u = "2017-10-13T08:59:35"
start_d = "2017-10-13T09:00:22"
end_d = "2017-10-13T09:02:00"

def norm_MC(b_data_u,b_data_d):
    """Calculate the shock normal vector using the Magnetic coplanarity(MC) method
    b_data_u: upstream magnetic field
    b_data_d: downstream magnetic field
    """
    b_up =  average_value(b_data_u) #average upstream magnetic field
    b_down = average_value(b_data_d) #average downstream magnetic field
    b_diff = b_down - b_up #change in magnetic field
    numerator = np.cross(np.cross(b_down,b_up),b_diff)
    denominator = np.linalg.norm(numerator)
    vector = numerator/denominator
    return vector

def norm_MX1(b_data_u,b_data_d,v_data_u,v_data_d):
     """Calculate the shock normal vector using the first mixed mode(MX1) method
     b_data_u: upstream magnetic field
     b_data_d: downstream magnetic field
     v_data_u: upstream velocity
     v_data_d: downstream velocity
     """
     b_up = average_value(b_data_u)
     b_down = average_value(b_data_d)
     b_diff = b_down - b_up
     v_up = average_value(v_data_u)
     v_down = average_value(v_data_d)
     v_diff = v_down - v_up
     numerator = np.cross(np.cross(b_up,v_diff),b_diff)
     denominator = np.linalg.norm(numerator)
     return numerator/denominator

def shock_velocity_MF(pv_data_u,pv_data_d,p_data_u,p_data_d,norm):
    """Calculate shock speed via mass flux(MF) algorithm
    pv_data_u: density*velocity upstream
    pv_data_d: density*velocity downstream
    p_data_u: density upstream
    p_data_d: density downstream
    norm:shock normal vector
    """
    pv_u = average_value(pv_data_u)
    pv_d = average_value(pv_data_d)
    pv_diff = pv_d - pv_u
    p_u = average_value(p_data_u)
    p_d = average_value(p_data_d)
    p_diff = p_d - p_u
    return abs(np.dot((pv_diff/p_diff),norm))

def shock_velocity_SB(b_data_u,b_data_d,v_data_u,v_data_d,norm):
    """Calculate shock velocity via Smith Burton(SB) algorithm
    b_data_u: upstream magnetic field
    b_data_d: downstream magnetic field
    v_data_u: upstream velocity
    v_data_d: downstream velocity
    """
    b_up = average_value(b_data_u)
    b_down = average_value(b_data_d)
    b_diff = b_down - b_up
    v_up = average_value(v_data_u)
    v_down = average_value(v_data_d)
    v_diff = v_down - v_up
    numerator = np.linalg.norm(np.cross(v_diff,b_down))
    denominator = np.linalg.norm(b_diff)
    return np.dot(v_up,norm) + numerator/denominator

def shock_obliquity(b_data_u,norm):
    """Calculate the angle between the shock normal and upstream magnetic field
    b_data_u: upstream magnetic field
    norm: shock normal vector
    """
    b_up = average_value(b_data_u)
    numerator = np.dot(b_up,norm)
    denominator = np.linalg.norm(b_up)*np.linalg.norm(norm)
    if np.arccos(numerator/denominator)*(180/np.pi) > 90:
        norm = -norm
        numerator = np.dot(b_up,norm)
        denominator = np.linalg.norm(b_up)*np.linalg.norm(norm)
    return np.arccos(numerator/denominator)*(180/np.pi)

def mach_number(b_data_u,v_data_u,p_data_u,norm,speed):
    """Calculate the Alfven Mach number using an approximated equation
    b_data_u: upstream magnetic field
    v_data_u: upstream velocity
    p_data_u: upstream density
    norm: shock normal vector
    """
    b_up = average_value(b_data_u)*10**(-9)
    v_up = average_value(v_data_u)*10**3
    p_up = average_value(p_data_u)*10**6
    nominator = np.linalg.norm(np.dot(v_up,norm)) - speed
    denominator = np.linalg.norm(b_up)/np.sqrt(sc.mu_0*sc.atomic_mass*p_up)
    return nominator/denominator

def overshoot(b_data,b_data_u):
    """
    Calculate overshoot compression ratio
    b_max: maximum magnetic field
    b_up: upstream magnetic field
    """
    b_max = np.nanmax(b_data)
    b_up = average_value(b_data_u)
    return b_max/b_up

#arrays of sliced data
b_tot_u = slicing_data(b_tot,start_u,end_u) #tot = magnitude
b_tot_d = slicing_data(b_tot,start_d,end_d)
b_gse_u = slicing_data(b_gse,start_u,end_u) #gse = vector
b_gse_d = slicing_data(b_gse,start_d,end_d)
ion_velocity_u = slicing_data(ion_velocity,start_u,end_u)
ion_velocity_d = slicing_data(ion_velocity,start_d,end_d)
ion_density_u = slicing_data(ion_density,start_u,end_u)
ion_density_d = slicing_data(ion_density,start_d,end_d)
ion_densityxvelocity_u = slicing_data(ion_densityxvelocity,start_u,end_u)
ion_densityxvelocity_d = slicing_data(ion_densityxvelocity,start_d,end_d)

#calculated parameters
shock_norm_MC = norm_MC(b_gse_u,b_gse_d)
shock_norm_MX1 = norm_MX1(b_gse_u,b_gse_d,ion_velocity_u,ion_velocity_d)
shock_v_MF_MC = shock_velocity_MF(ion_densityxvelocity_u,ion_densityxvelocity_d,ion_density_u,ion_density_d,shock_norm_MC)
shock_v_MF_MX1 = shock_velocity_MF(ion_densityxvelocity_u,ion_densityxvelocity_d,ion_density_u,ion_density_d,shock_norm_MX1)
shock_v_shock_SB = shock_velocity_SB(b_gse_u,b_gse_d,ion_velocity_u,ion_velocity_d,shock_norm_MX1)
shock_angle_MC = shock_obliquity(b_gse_u,shock_norm_MC)
shock_angle_MX1 = shock_obliquity(b_gse_u,shock_norm_MX1)
Mach_number_MC = mach_number(b_tot_u,ion_velocity_u,ion_density_u,shock_norm_MC,shock_v_MF_MC)
Mach_number_MX1 = mach_number(b_tot_u,ion_velocity_u,ion_density_u,shock_norm_MX1,shock_v_MF_MX1)
Comp_overshoot = overshoot(b_tot,b_tot_u)

#ensures that the shock normal vector is pointing upstream (+x direction)
shock_norm_MC[0] = abs(shock_norm_MC[0])
shock_norm_MX1[0] = abs(shock_norm_MX1[0])

#display parameters
print("{:^27}| MC:{:}, MX1:{}".format("Shock norm",shock_norm_MC,shock_norm_MX1))
print("{:^27}| MF-MC: {:.3f}, MF-MX1: {:.3f}, SB: {:.3f}".format("Shock velocity(kms\u207B\u00B9)",shock_v_MF_MC,shock_v_MF_MX1,shock_v_shock_SB))
print("{:^27}| MC: {:.3f}, Shock angle MX1: {:.3f}".format("Shock angle",shock_angle_MC,shock_angle_MX1))
print("{:^27}| MC:{:.3f}, MX1: {:.3f}".format("M_A",Mach_number_MC,Mach_number_MX1))
print("{:^27}| {:.3f}".format("Compression overshoot ratio",Comp_overshoot))

In [ ]:
#user input
time_start = "2017-10-13T09:01:00"
time_end = "2017-10-13T09:01:30"
fit_start = 13
fit_end = 30
energy_bins = np.array([2.16,3.91,7.07,10.93,14.47,19.16,25.37,33.59,44.48,58.89,77.98,
               103.24,136.7,180.99,239.63,317.28,420.09,556.22,736.45,975.08,
               1291.03,1709.37,2263.26,2996.62,3967.62,5253.24,6955.46,9209.24,
               12193.31,16144.31,21375.56,28301.89])*10**(-3) #FPI energy channels measured in KeV
#FPI noise level 
FPI_phase_space_density_noise = np.array([ 81.0995596, 42.4075837, 23.1898909, 14.3478487])
FPI_energy_noise = np.array([12193.31,16144.31,21375.56,28301.89])*10**(-3)
FPI_std_noise = np.array([ 0.699723902, 0.501129747, 0.376583384, 0.364246610])
#FEEPs noise level measured in KeV
FEEPs_phase_space_density_noise = np.array([np.nan,0.12909757, 0.07921698, 0.03887495, 0.03683296, 0.03087287, 0.02231999,
 0.0195845,  0.01363228, 0.01022196, 0.00585628, 0.00933458, 0.00674828,
 0.00380524, 0.00156136])
FEEPs_energies_noise = np.array([np.nan,76.8,  95.4, 114.1, 133.,  153.7, 177.6, 205.1, 236.7, 273.2, 315.4, 363.8, 419.7,
 484.2, 558.6])
FEEPs_noise_std = np.array([np.nan,0.03535483, 0.04724885, 0.00632242, 0.014478,   0.01197523, 0.,
 0.0069682,  0.00839755, 0.00312024, 0.00168536, 0.00305993, 0.,
 0.00110779, 0.])*3 #use 3x the FEEPs noise std to ensure clearance of false data
#time interval indexes
time_start_index = np.where(time.astype("datetime64[s]") == np.datetime64(time_start))[0][0] #array index of start time
time_end_index = np.where(time.astype("datetime64[s]") == np.datetime64(time_end))[0][0] #array index of end time
FEEPs_time_start_index = np.where(FEEPs_time_corr.astype("datetime64[s]") == np.datetime64(time_start))[0][0]
FEEPs_time_end_index = np.where(FEEPs_time_corr.astype("datetime64[s]") == np.datetime64(time_end))[0][0]

def avg_flux_distribution(flux,start,end):
    """Averages the flux over time for each energy channel
    flux: flux-time matrix
    start: start time
    end: end time
    """
    mini_matrix = flux[start:end+1,:] #slice ion energy matrix for desired range time
    return np.mean(mini_matrix,axis=0)

def FEEP_avg_flux_distribution(flux,start,end):
    """Averages the flux over time for each energy channel
    flux: flux-time matrix
    start: start time
    end: end time
    """
    mini_matrix = flux[start:end+1,:] #slice ion energy matrix for desired range time
    mini_matrix[mini_matrix == 0] = np.nan #substitute 0 for nan so matplotlib ignores them on the plot
    average_flux = np.nanmean(mini_matrix,axis = 0) #np.nanmean ignores nan in the calculation of the mean or returns nan if all elements are nan
    mask = ~np.isnan(average_flux)
    return average_flux[mask],FEEPs_energies[mask] #removing the energy channels without any flux

def PSD(energy_flux,channels):
    """Convert differential flux into phase space density"""
    return 0.99*(energy_flux/(channels*(channels+2*938.25)))

def interpolation():
    return sci.interp1d(energy_range,PSD_range,kind = "linear")

def fitting(start,end):
    """Calculates the linear curve tangent to a section of the curve"""
    f = interpolation() #generates a smoother curve for analysis
    energy_fine = np.linspace(energy_range.min(),energy_range.max(),20000)
    flux_fine = f(energy_fine)
    m,b = np.polyfit(np.log10(energy_fine[(energy_fine >= start) & (energy_fine <= end)]),np.log10(flux_fine[(energy_fine >= start) & (energy_fine <= end)]),1) #coefficients of linear line at segment 
    x = energy_fine[(energy_fine >= 1) & (energy_fine <= end+(end/2))]
    
    A = 10**b
    y = A*x**m
    print("p = {}".format(m))
    plt.plot(x,y,color = "purple",linewidth = 3, label = "p = {:.3f}".format(m))
    return 

def minimum_energy():
    """Creates a dashed vertical line to indicate the suprathermal ion region by 
    using the ion energy in the upstream as the minimum"""
    
    f = interpolation() #generates a smoother curve for analysis
    energy_fine = np.linspace(energy_range.min(),energy_range.max(),20000)
    flux_fine = f(energy_fine)
    peak_energy_range = energy_fine[(0.5 <= energy_fine) & (energy_fine <= 2)]
    peak_flux_range = flux_fine[(0.5 <= energy_fine) & (energy_fine <= 2)]
    peak = np.where(peak_flux_range == np.max(peak_flux_range))
    peak_energy = 10*peak_energy_range[peak]
    plt.axvline(peak_energy, color = "black",ls = "--", label ="10E$_S$$_W$")
    return 

avg_flux = avg_flux_distribution(ion_flux,time_start_index,time_end_index)  #find the average flux of each energy channel over time,in other words averaging over rows for each column
avg_flux_corr = avg_flux/energy_bins #energy flux to flux conversion
phase_space_density = PSD(avg_flux_corr,energy_bins*10**(-3))
f = [10**9.6*x**(-2) for x in energy_bins*10**3] #FPI noise level,energy is measured in eV and energy_bins is in KeV

avg_FEEPs_flux,FEEPs_energies_corr = FEEP_avg_flux_distribution(FEEPs_intensities,FEEPs_time_start_index,FEEPs_time_end_index)  
FEEPs_phase_space_density = PSD(avg_FEEPs_flux,FEEPs_energies_corr*10**(-3)) #need 10^-3 as PSD formula needs energy in MeV
PSD_range = np.concatenate([phase_space_density,FEEPs_phase_space_density]) #concatenating the PSD and energy of FPI and FEEPs to generate the interpolation curve
energy_range = np.concatenate([energy_bins,FEEPs_energies_corr])
linear_fit_coefficients = fitting(fit_start,fit_end)
minimum_energy()

plt.plot(energy_bins,phase_space_density,color = "green",marker = ".",label = "FPI")
plt.plot(FEEPs_energies_corr,FEEPs_phase_space_density,color = "blue",marker = ".",label = "FEEPs")
plt.plot(energy_bins,f,label = "FPI noise")
plt.errorbar(FEEPs_energies_noise,FEEPs_phase_space_density_noise,yerr = [np.zeros_like(FEEPs_noise_std),FEEPs_noise_std],color = "red",marker = ".",label = "FEEPs noise")
#plt.errorbar(FPI_energy_noise,FPI_phase_space_density_noise,yerr = [np.zeros_like(FPI_std_noise),FPI_std_noise],color = "orange",marker = ".", label = "FPI noise")
plt.xlabel("Energy (keV)", size = 13)
plt.ylabel("Phase space density (s\u00B3/km\u2076)", size = 13)
plt.xscale("log")
plt.yscale("log")
plt.ylim(0,1E10)
plt.title("Ion PSD distribution at {} ({}-{})".format(time[time_start_index].astype("datetime64[D]"),time[time_start_index].astype("datetime64[s]").astype(object).time(),time[time_end_index].astype("datetime64[s]").astype(object).time()), size = 13)
plt.grid(True)
plt.legend()
plt.xlim(0,1000)
plt.ylim(0,1E10)
plt.tick_params(axis = "both", labelsize = 13)
plt.figure()
FEEPs_phase_space_density_noise[FEEPs_phase_space_density_noise == np.nan] = 0 #set nan values to 0 for the noise corrected steps (subtracting nan returns nan)

phase_space_density_noise_corr = np.array([a - b for a,b in zip(phase_space_density,f)]) #subtract FPI noise from PSD
FEEPs_phase_space_density_noise_corr = np.array([a - b for a,b in zip(FEEPs_phase_space_density,FEEPs_phase_space_density_noise)]) #subtract FEEPs noise from FEEPs PSD

#Remove any FEEPs or FPI points that fall below the noise level since they drop below zero and go off the graph
phase_space_density_noise_corr[phase_space_density_noise_corr <= 0] = np.nan
mask = ~np.isnan(phase_space_density_noise_corr)
phase_space_density_noise_corr = phase_space_density_noise_corr[mask]
energy_bins = energy_bins[mask]
f = [10**9.6*x**(-2) for x in energy_bins*10**3]


FEEPs_phase_space_density_noise_corr[FEEPs_phase_space_density_noise_corr <= 0] = np.nan
mask = ~np.isnan(FEEPs_phase_space_density_noise_corr)
FEEPs_phase_space_density_noise_corr = FEEPs_phase_space_density_noise_corr[mask]
FEEPs_energies_corr = FEEPs_energies_corr[mask]

PSD_range = np.concatenate([phase_space_density_noise_corr,FEEPs_phase_space_density_noise_corr])
energy_range = np.concatenate([energy_bins,FEEPs_energies_corr])
linear_fit_coefficients = fitting(fit_start,fit_end)
minimum_energy()

plt.plot(energy_bins,phase_space_density_noise_corr,color = "green",marker = ".",label = "FPI")
plt.plot(FEEPs_energies_corr,FEEPs_phase_space_density_noise_corr,color = "blue",marker = ".",label = "FEEPs")
plt.xlabel("Energy (keV)",size = 13)
plt.ylabel("Phase space density (s\u00B3/km\u2076)", size = 13)
plt.xscale("log")
plt.yscale("log")
plt.title("Noise corrected Ion PSD distribution at {} ({}-{})".format(time[time_start_index].astype("datetime64[D]"),time[time_start_index].astype("datetime64[s]").astype(object).time(),time[time_end_index].astype("datetime64[s]").astype(object).time()),size = 13)
plt.grid(True)
plt.legend()
plt.xlim(0,1000)
plt.ylim(0,1E10)
print(FEEPs_energies)
print(energy_bins)

In [ ]:
FEEPs_phase_space_density_noise[FEEPs_phase_space_density_noise == np.nan] = 0 #set nan values to 0 for the noise corrected steps (subtracting nan returns nan)

phase_space_density_noise_corr = np.array([a - b for a,b in zip(phase_space_density,f)]) #subtract FPI noise from PSD
FEEPs_phase_space_density_noise_corr = np.array([a - b for a,b in zip(FEEPs_phase_space_density,FEEPs_phase_space_density_noise)]) #subtract FEEPs noise from FEEPs PSD

#Remove any FEEPs or FPI points that fall below the noise level since they drop below zero and go off the graph
phase_space_density_noise_corr[phase_space_density_noise_corr <= 0] = np.nan
mask = ~np.isnan(phase_space_density_noise_corr)
phase_space_density_noise_corr = phase_space_density_noise_corr[mask]
energy_bins = energy_bins[mask]
f = [10**9.6*x**(-2) for x in energy_bins*10**3]


FEEPs_phase_space_density_noise_corr[FEEPs_phase_space_density_noise_corr <= 0] = np.nan
mask = ~np.isnan(FEEPs_phase_space_density_noise_corr)
FEEPs_phase_space_density_noise_corr = FEEPs_phase_space_density_noise_corr[mask]
FEEPs_energies_corr = FEEPs_energies_corr[mask]

PSD_range = np.concatenate([phase_space_density_noise_corr,FEEPs_phase_space_density_noise_corr])
energy_range = np.concatenate([energy_bins,FEEPs_energies_corr])
linear_fit_coefficients = fitting(fit_start,fit_end)
minimum_energy()

plt.plot(energy_bins,phase_space_density_noise_corr,color = "green",marker = ".",label = "FPI")
plt.plot(FEEPs_energies_corr,FEEPs_phase_space_density_noise_corr,color = "blue",marker = ".",label = "FEEPs")



plt.xlabel("Energy (keV)")
plt.ylabel("Phase space density (s\u00B3/km\u2076)")
plt.xscale("log")
plt.yscale("log")
plt.title("Noise corrected Ion energy-flux distribution at {} ({}-{})".format(time[time_start_index].astype("datetime64[D]"),time[time_start_index].astype("datetime64[s]").astype(object).time(),time[time_end_index].astype("datetime64[s]").astype(object).time()))
plt.grid(True)
plt.legend()


In [ ]:
"""Calculating the FPI noise level"""
time_start = "2022-02-03T10:45:00"
time_end = "2022-02-03T10:46:00"
time_start_index = np.where(time.astype("datetime64[s]") == np.datetime64(time_start))[0][0] #array index of start time
time_end_index = np.where(time.astype("datetime64[s]") == np.datetime64(time_end))[0][0] #array index of end time
energy_bins = np.array([2.16,3.91,7.07,10.93,14.47,19.16,25.37,33.59,44.48,58.89,77.98,
               103.24,136.7,180.99,239.63,317.28,420.09,556.22,736.45,975.08,
               1291.03,1709.37,2263.26,2996.62,3967.62,5253.24,6955.46,9209.24,
               12193.31,16144.31,21375.56,28301.89])*10**(-3) #measured in KeV

def avg_flux_distribution(flux,start,end):
    """Averages the flux over time for each energy channel
    flux: flux-time matrix
    start: start time
    end: end time
    """
    mini_matrix = flux[start:end+1,:] #slice ion energy matrix for desired range time
    return np.mean(mini_matrix,axis=0),np.std(mini_matrix,axis = 0)

def PSD(energy_flux,channels):
    return 0.99*(energy_flux/(channels*(channels+2*938.25)))

avg_flux,FPI_std = avg_flux_distribution(ion_flux,time_start_index,time_end_index)  #find the average flux of each energy channel over time,in other words averaging over rows for each column
avg_flux_corr = avg_flux/energy_bins #energy flux to flux conversion
phase_space_density = PSD(avg_flux_corr,energy_bins*10**(-3))
FPI_noise_std = PSD(FPI_std,energy_bins)
print(phase_space_density)
print(FPI_noise_std)


In [ ]:
"""Calculating the FEEPs noise level"""
time_start = "2022-02-03T10:45:00"
time_end = "2022-02-03T10:46:00"
FEEPs_time_start_index = np.where(FEEPs_time_corr.astype("datetime64[s]") == np.datetime64(time_start))[0][0]
FEEPs_time_end_index = np.where(FEEPs_time_corr.astype("datetime64[s]") == np.datetime64(time_end))[0][0]

def FEEP_avg_flux_distribution(flux,start,end):
    """Averages the flux over time for each energy channel
    flux: flux-time matrix
    start: start time
    end: end time
    """
    mini_matrix = flux[start:end+1,:] #slice ion energy matrix for desired range time
    mini_matrix[mini_matrix == 0] = np.nan #change 0 to nan
    average_flux = np.nanmean(mini_matrix,axis = 0) #np.nanmean ignores nan in the calculation of the mean or returns nan if all elements are nan
    standard_dev = np.nanstd(mini_matrix,axis = 0)
    mask = ~np.isnan(average_flux)
    return average_flux[mask],FEEPs_energies[mask],standard_dev[mask] #removing the energy channels without any flux


def PSD(energy_flux,channels):
    return 0.99*(energy_flux/(channels*(channels+2*938.25)))



avg_FEEPs_flux_noise,FEEPs_energies_corr_noise,FEEPs_noise_std = FEEP_avg_flux_distribution(FEEPs_intensities,FEEPs_time_start_index,FEEPs_time_end_index)  #substitute 0 for nan so matplotlib ignores them on the plot
FEEPs_noise_std = PSD(FEEPs_noise_std,FEEPs_energies_corr_noise*10**(-3)) #convert flux std to PSD std of the noise level
FEEPs_phase_space_density_noise = PSD(avg_FEEPs_flux_noise,FEEPs_energies_corr_noise*10**(-3)) #need 10^-3 as PSD formula needs energy in MeV
print(FEEPs_phase_space_density_noise,FEEPs_energies_corr_noise,FEEPs_noise_std)


In [ ]:
"""Analysing data on graphs"""
excel_path = "C:/Users/antho/OneDrive/Desktop/Bsc project/MMS1 shock crossings .xlsx" #File path of csv (data) file
file = pd.read_excel(io = excel_path,header = [0,1],sheet_name= 0) #Read csv file
data_file = pd.DataFrame(file) #Convert to Panda dataframe
new_cols = []
for top, sub in data_file.columns:
    if str(top).startswith("Unnamed"):
        new_cols.append((sub, ""))   # promote second-row name
    else:
        new_cols.append((top, sub))

data_file.columns = pd.MultiIndex.from_tuples(new_cols)
print(data_file.columns)
time = data_file["Date"]
mach = np.array(data_file["Alfven Mach"]["MX1"])
p = np.array(data_file["Alfven Mach"]["p"])
angle = np.array(data_file["Shock obliquity(degrees)"]["MX1"])
e_max = np.array(data_file["Alfven Mach"]["E_MAX (keV)"])

plt.scatter(angle,p,c = mach, cmap = "plasma",marker = "x") #Colour map of p against angle, with Mach as colour gradient
plt.colorbar(label = "Alfven Mach number")
plt.xlabel("Angle (degrees)")
plt.ylabel("power law slope")
plt.xlim(0,90)
plt.grid(True)
plt.ylim(-8,-2)
plt.title("Shock acceleration efficiency across different shock angles")
plt.legend()
for i in range(0,90,15): #Finding the median of bin sizes 15 degrees and plotting dashed lines on plot
    max = i + 15
    middle = i + 7.5
    mask = (angle > i) & (angle < max)
    angle2 = angle[mask]
    p2 = p[mask]
    median = np.median(p2)
    l_quartile = median - np.quantile(p2,0.25) #value of lower error bar (must be a positive value)
    u_quartile = np.quantile(p2,0.75) - median
    plt.hlines(median,i,max,color = "black", linestyles= "--", label = "average")
    plt.errorbar(middle,median,yerr = [[l_quartile],[u_quartile]],color = "black",capsize= 6,capthick= 1.5 )
    end

plt.figure()
plt.hist(mach,bins = [1,2,3,4,5,6,7]) #Histogram of Mach number
plt.xlabel("Alfven Mach number (Ma)")
plt.ylabel("Number of shock crossings")
plt.title("Distribution of Mach number")
plt.xlim(1,7)

for i in range(0,90,30): #Histograms of maximum energy for different angle bins
    plt.figure()
    max = i + 30
    mask = (angle > i) & (angle < max)
    e_max2 = e_max[mask]
    plt.hist(e_max2, bins = [9.2,30,57.9,76.8,95.4,114.1,133,153.7,177.6,205.1,236.7,273.2,315.4,363.8])
    plt.xlabel("Maximum energy (keV)", size = 13)
    plt.ylabel("Number of crossings", size = 13)
    plt.title("Max energy distribution for shock angles {} - {}".format(i,max))
    
    plt.ylim(0,13)